# Brazilian E-Commerce Public Dataset by Olist

## Load the Data

In [ ]:
import pandas as pd
import numpy as np
import os

# ============================================================
# STEP 0 — LOAD ALL OLIST TABLES
# ============================================================

customers = pd.read_csv('/content/olist_customers.csv')

geolocation = pd.read_csv('/content/olist_geolocation.csv')

items = pd.read_csv('/content/olist_order_items.csv')

payments = pd.read_csv('/content/olist_order_payments.csv')

reviews = pd.read_csv('/content/olist_order_reviews.csv')

orders = pd.read_csv('/content/olist_orders.csv',
    parse_dates=[
        'order_purchase_timestamp',
        'order_approved_at',
        'order_delivered_carrier_date',
        'order_delivered_customer_date',
        'order_estimated_delivery_date'
    ]
)

products = pd.read_csv('/content/olist_products.csv')

sellers = pd.read_csv('/content/olist_sellers.csv')

translation = pd.read_csv(
    '/content/product_categoryname_translation.csv'
)


## STEP 1 — UNDERSTAND THE DATA

In [ ]:
tables = {
    'customers': customers,
    'geolocation': geolocation,
    'items': items,
    'payments': payments,
    'reviews': reviews,
    'orders': orders,
    'products': products,
    'sellers': sellers,
    'translation': translation
}

for name, df in tables.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
geolocation: (1000163, 5)
items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
orders: (99441, 8)
products: (32951, 9)
sellers: (3095, 4)
translation: (71, 2)


check columns:

In [ ]:
for name, df in tables.items():
    print(f"\n========== {name.upper()} ==========")
    print(df.columns.tolist())


========== CUSTOMERS ==========
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

========== GEOLOCATION ==========
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

========== ITEMS ==========
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

========== PAYMENTS ==========
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

========== REVIEWS ==========
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

========== ORDERS ==========
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

========== PRODUCTS ==========
['product_id', 'product_category_n

STEP 2 — CHECK DATA TYPES

In [ ]:
for name, df in tables.items():
    print(f"\n========== {name.upper()} ==========")
    print(df.dtypes)


========== CUSTOMERS ==========
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object

========== GEOLOCATION ==========
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object

========== ITEMS ==========
order_id                object
order_item_id            int64
product_id              object
seller_id               object
shipping_limit_date     object
price                  float64
freight_value          float64
dtype: object

========== PAYMENTS ==========
order_id                 object
payment_sequential        int64
payment_type             object
payment_installments      int64
payment_value           float64
dtype: object

========== REVIEWS ==========
review_id                  object
ord

STEP 3 — CHECK NULL VALUES

In [ ]:
for name, df in tables.items():

    print(f"\n========== {name.upper()} ==========")

    null_summary = pd.DataFrame({
        'null_count': df.isnull().sum(),
        'null_percentage': (
            df.isnull().sum() / len(df) * 100
        ).round(2)
    })

    print(null_summary[null_summary['null_count'] > 0])


========== CUSTOMERS ==========
Empty DataFrame
Columns: [null_count, null_percentage]
Index: []

========== GEOLOCATION ==========
Empty DataFrame
Columns: [null_count, null_percentage]
Index: []

========== ITEMS ==========
Empty DataFrame
Columns: [null_count, null_percentage]
Index: []

========== PAYMENTS ==========
Empty DataFrame
Columns: [null_count, null_percentage]
Index: []

========== REVIEWS ==========
                        null_count  null_percentage
review_comment_title         87656            88.34
review_comment_message       58247            58.70

========== ORDERS ==========
                               null_count  null_percentage
order_approved_at                     160             0.16
order_delivered_carrier_date         1783             1.79
order_delivered_customer_date        2965             2.98

========== PRODUCTS ==========
                            null_count  null_percentage
product_category_name              610             1.85
product_name_l

STEP 4 — CHECK DUPLICATES

In [ ]:
for name, df in tables.items():

    print(
        f"{name}: {df.duplicated().sum()} duplicate rows"
    )

customers: 0 duplicate rows
geolocation: 261831 duplicate rows
items: 0 duplicate rows
payments: 0 duplicate rows
reviews: 0 duplicate rows
orders: 0 duplicate rows
products: 0 duplicate rows
sellers: 0 duplicate rows
translation: 0 duplicate rows


STEP 5 — CUSTOMERS TABLE

In [ ]:
print(
    "Duplicate customer_id:",
    customers['customer_id'].duplicated().sum()
)

Duplicate customer_id: 0


In [ ]:
print(
    "Duplicate customer_unique_id:",
    customers['customer_unique_id'].duplicated().sum()
)

Duplicate customer_unique_id: 3345


In [ ]:
customers.isnull().sum()

,0
customer_id,0
customer_unique_id,0
customer_zip_code_prefix,0
customer_city,0
customer_state,0


clean Text:

In [ ]:
customers['customer_city'] = (
    customers['customer_city']
    .str.strip()
    .str.lower()
)

customers['customer_state'] = (
    customers['customer_state']
    .str.strip()
    .str.upper()
)

STEP 6 — GEOLOCATION TABLE

In [ ]:
print(
    "Duplicate rows:",
    geolocation.duplicated().sum()
)

Duplicate rows: 261831


The geolocation dataset can legitimately contain repeated ZIP-prefix/location combinations, so don't blindly delete them.

check numeric columns:

In [ ]:
geolocation[['geolocation_lat', 'geolocation_lng']].describe()

,geolocation_lat,geolocation_lng
count,1.000163e+06,1.000163e+06
mean,-2.117615e+01,-4.639054e+01
std,5.715866e+00,4.269748e+00
min,-3.660537e+01,-1.014668e+02
25%,-2.360355e+01,-4.857317e+01
50%,-2.291938e+01,-4.663788e+01
75%,-1.997962e+01,-4.376771e+01
max,4.506593e+01,1.211054e+02


Check invalid latitude

In [ ]:
geolocation[
    (geolocation['geolocation_lat'] < -90) |
    (geolocation['geolocation_lat'] > 90)
]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state


Check invalid longitude

In [ ]:
geolocation[
    (geolocation['geolocation_lng'] < -180) |
    (geolocation['geolocation_lng'] > 180)
]

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state


Clean state

In [ ]:
geolocation['geolocation_state'] = (
    geolocation['geolocation_state']
    .str.strip()
    .str.upper()
)

STEP 7 — ORDER ITEMS TABLE

In [ ]:
items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


In [ ]:
items.isnull().sum()

,0
order_id,0
order_item_id,0
product_id,0
seller_id,0
shipping_limit_date,0
price,0
freight_value,0


In [ ]:
items.duplicated().sum()

np.int64(0)

In [ ]:
print(
    "Missing order_id:",
    items['order_id'].isnull().sum()
)

print(
    "Missing product_id:",
    items['product_id'].isnull().sum()
)

Missing order_id: 0
Missing product_id: 0


In [ ]:
items['price'].describe()

,price
count,112650.000000
mean,120.653739
std,183.633928
min,0.850000
25%,39.900000
50%,74.990000
75%,134.900000
max,6735.000000


In [ ]:
items[items['price'] <= 0]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [ ]:
items[items['freight_value'] < 0]

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


In [ ]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


STEP 8 — PAYMENTS TABLE

In [ ]:
payments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


In [ ]:
payments.isnull().sum()

,0
order_id,0
payment_sequential,0
payment_type,0
payment_installments,0
payment_value,0


In [ ]:
payments.duplicated().sum()

np.int64(0)

In [ ]:
payments['payment_type'].value_counts()

,count
payment_type,
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3


In [ ]:
payments['payment_value'].describe()

,payment_value
count,103886.000000
mean,154.100380
std,217.494064
min,0.000000
25%,56.790000
50%,100.000000
75%,171.837500
max,13664.080000


In [ ]:
payments[
    payments['payment_value'] <= 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


In [ ]:
payments['payment_installments'].describe()

,payment_installments
count,103886.000000
mean,2.853349
std,2.687051
min,0.000000
25%,1.000000
50%,1.000000
75%,4.000000
max,24.000000


In [ ]:
payments[
    payments['payment_installments'] <= 0
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


In [ ]:
payments['payment_type'] = (
    payments['payment_type']
    .str.strip()
    .str.lower()
)

STEP 9 — REVIEWS TABLE

In [ ]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


In [ ]:
reviews.isnull().sum()

,0
review_id,0
order_id,0
review_score,0
review_comment_title,87656
review_comment_message,58247
review_creation_date,0
review_answer_timestamp,0


In [ ]:
reviews.duplicated().sum()

np.int64(0)

In [ ]:
reviews['review_id'].duplicated().sum()

np.int64(814)

In [ ]:
reviews['review_score'].value_counts().sort_index()

,count
review_score,
1,11424
2,3151
3,8179
4,19142
5,57328


In [ ]:
reviews[
    ~reviews['review_score'].isin([1, 2, 3, 4, 5])
]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [ ]:
reviews['review_creation_date'] = pd.to_datetime(
    reviews['review_creation_date'],
    errors='coerce'
)

reviews['review_answer_timestamp'] = pd.to_datetime(
    reviews['review_answer_timestamp'],
    errors='coerce'
)

In [ ]:
reviews[
    reviews['review_answer_timestamp']
    < reviews['review_creation_date']
]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp


In [ ]:
reviews['review_comment_title'] = (
    reviews['review_comment_title']
    .fillna('No Title')
)

reviews['review_comment_message'] = (
    reviews['review_comment_message']
    .fillna('No Comment')
)

STEP 10 — ORDERS TABLE

In [ ]:
orders['order_status'].value_counts()

,count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


In [ ]:
orders['order_id'].duplicated().sum()

np.int64(0)

In [ ]:
# Keep delivered orders
# If your analysis is specifically about delivered orders:

orders = orders[
    orders['order_status'] == 'delivered'
].copy()

Data Consistency:


In [ ]:
orders[
    orders['order_approved_at']
    < orders['order_purchase_timestamp']
]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [ ]:
# Create delivery days
orders['delivery_days'] = (
    orders['order_delivered_customer_date']
    - orders['order_purchase_timestamp']
).dt.total_seconds() / 86400

orders['delivery_days'] = (
    orders['delivery_days'].round(2)
)

In [ ]:
# Late delivery
orders['is_late'] = np.where(
    orders['order_delivered_customer_date']
    > orders['order_estimated_delivery_date'],
    1,
    0
)

Date Dimensions:


In [ ]:
orders['order_date'] = (
    orders['order_purchase_timestamp'].dt.date
)

orders['order_year'] = (
    orders['order_purchase_timestamp'].dt.year
)

orders['order_month'] = (
    orders['order_purchase_timestamp'].dt.month
)

orders['order_month_name'] = (
    orders['order_purchase_timestamp'].dt.month_name()
)

orders['order_quarter'] = (
    orders['order_purchase_timestamp'].dt.quarter
)

orders['order_dayofweek'] = (
    orders['order_purchase_timestamp'].dt.day_name()
)

In [ ]:
import pandas as pd

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]


# ============================================================
# CLEAN INVALID DATE VALUES
# ============================================================

for col in date_columns:

    if col in orders.columns:

        # Convert everything to string first
        orders[col] = orders[col].astype("string").str.strip()

        # Replace all known invalid/missing date formats with NA
        orders[col] = orders[col].replace(
            [
                "0000-00-00",
                "0000-00-00 00:00:00",
                "0000-00-00 00:00:00.000000",
                "",
                "nan",
                "NaN",
                "None",
                "NULL",
                "null"
            ],
            pd.NA
        )

        # Convert valid values to datetime
        # Invalid values automatically become NaT
        orders[col] = pd.to_datetime(
            orders[col],
            errors="coerce"
        )


# ============================================================
# CHECK INVALID / MISSING DATES BEFORE SAVING
# ============================================================

print("\nDate Cleaning Results")
print("=" * 60)

for col in date_columns:

    if col in orders.columns:

        # Count missing dates
        missing_count = orders[col].isna().sum()

        # Count invalid zero dates
        zero_date_count = (
            orders[col]
            .astype("string")
            .str.contains("0000-00-00", na=False)
            .sum()
        )

        print(f"{col}")
        print(f"  Missing dates : {missing_count}")
        print(f"  Zero dates    : {zero_date_count}")
        print()


# ============================================================
# CONVERT DATETIME TO MYSQL-FRIENDLY FORMAT
# ============================================================

for col in date_columns:

    if col in orders.columns:

        orders[col] = orders[col].dt.strftime(
            "%Y-%m-%d %H:%M:%S"
        )

        # Convert missing dates to empty fields
        orders[col] = orders[col].fillna("")


# ============================================================
# FINAL CHECK
# ============================================================

print("=" * 60)
print("FINAL CHECK")
print("=" * 60)

for col in date_columns:

    if col in orders.columns:

        invalid_zero_dates = (
            orders[col]
            .astype("string")
            .str.contains("0000-00-00", na=False)
            .sum()
        )

        print(
            f"{col}: "
            f"{invalid_zero_dates} zero-date values"
        )


# ============================================================
# DISPLAY SAMPLE
# ============================================================

print("\nSample of cleaned date columns:")

print(
    orders[date_columns].head(10)
)



Date Cleaning Results
order_purchase_timestamp
  Missing dates : 0
  Zero dates    : 0

order_approved_at
  Missing dates : 14
  Zero dates    : 0

order_delivered_carrier_date
  Missing dates : 2
  Zero dates    : 0

order_delivered_customer_date
  Missing dates : 8
  Zero dates    : 0

order_estimated_delivery_date
  Missing dates : 0
  Zero dates    : 0

FINAL CHECK
order_purchase_timestamp: 0 zero-date values
order_approved_at: 0 zero-date values
order_delivered_carrier_date: 0 zero-date values
order_delivered_customer_date: 0 zero-date values
order_estimated_delivery_date: 0 zero-date values

Sample of cleaned date columns:
   order_purchase_timestamp    order_approved_at order_delivered_carrier_date  \
0       2017-10-02 10:56:33  2017-10-02 11:07:15          2017-10-04 19:55:00   
1       2018-07-24 20:41:37  2018-07-26 03:24:27          2018-07-26 14:31:00   
2       2018-08-08 08:38:49  2018-08-08 08:55:23          2018-08-08 13:50:00   
3       2017-11-18 19:28:06  2017-11-1

In [ ]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,is_late,order_date,order_year,order_month,order_month_name,order_quarter,order_dayofweek
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,8.44,0,2017-10-02,2017,10,October,4,Monday
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,13.78,0,2018-07-24,2018,7,July,3,Tuesday
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,9.39,0,2018-08-08,2018,8,August,3,Wednesday
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,13.21,0,2017-11-18,2017,11,November,4,Saturday
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,2.87,0,2018-02-13,2018,2,February,1,Tuesday


STEP 11 — PRODUCTS TABLE

In [ ]:
products.isnull().sum()

,0
product_id,0
product_category_name,610
product_name_lenght,610
product_description_lenght,610
product_photos_qty,610
product_weight_g,2
product_length_cm,2
product_height_cm,2
product_width_cm,2


In [ ]:
products['product_id'].duplicated().sum()

np.int64(0)

Numeric columns

In [ ]:
product_numeric_cols = [
    'product_name_lenght',
    'product_description_lenght',
    'product_photos_qty',
    'product_weight_g',
    'product_length_cm',
    'product_height_cm',
    'product_width_cm'
]

products[product_numeric_cols].describe()

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
count,32341.000000,32341.000000,32341.000000,32949.000000,32949.000000,32949.000000,32949.000000
mean,48.476949,771.495285,2.188986,2276.472488,30.815078,16.937661,23.196728
std,10.245741,635.115225,1.736766,4282.038731,16.914458,13.637554,12.079047
min,5.000000,4.000000,1.000000,0.000000,7.000000,2.000000,6.000000
25%,42.000000,339.000000,1.000000,300.000000,18.000000,8.000000,15.000000
50%,51.000000,595.000000,1.000000,700.000000,25.000000,13.000000,20.000000
75%,57.000000,972.000000,3.000000,1900.000000,38.000000,21.000000,30.000000
max,76.000000,3992.000000,20.000000,40425.000000,105.000000,105.000000,118.000000


In [ ]:
# Check negative values
for col in product_numeric_cols:

    print(
        col,
        "negative:",
        (products[col] < 0).sum()
    )

product_name_lenght negative: 0
product_description_lenght negative: 0
product_photos_qty negative: 0
product_weight_g negative: 0
product_length_cm negative: 0
product_height_cm negative: 0
product_width_cm negative: 0


Missing Categorys:


In [ ]:
products['product_category_name'] = (
    products['product_category_name']
    .fillna('unknown')
)

FIx spelling:

In [ ]:
products.rename(
    columns={
        'product_name_lenght':
            'product_name_length',

        'product_description_lenght':
            'product_description_length'
    },
    inplace=True
)

In [ ]:
products['product_category_name'].value_counts()

,count
product_category_name,
cama_mesa_banho,3029
esporte_lazer,2867
moveis_decoracao,2657
beleza_saude,2444
utilidades_domesticas,2335
...,...
fashion_roupa_infanto_juvenil,5
casa_conforto_2,5
pc_gamer,3


STEP 12 — SELLERS TABLE

In [ ]:
sellers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


In [ ]:
sellers.isnull().sum()

,0
seller_id,0
seller_zip_code_prefix,0
seller_city,0
seller_state,0


In [ ]:
sellers['seller_id'].duplicated().sum()

np.int64(0)

In [ ]:
sellers['seller_city'] = (
    sellers['seller_city']
    .str.strip()
    .str.lower()
)

sellers['seller_state'] = (
    sellers['seller_state']
    .str.strip()
    .str.upper()
)

STEP 13 — CATEGORY TRANSLATION TABLE

In [ ]:
translation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


In [ ]:
translation.isnull().sum()

,0
product_category_name,0
product_category_name_english,0


In [ ]:
translation.duplicated().sum()

np.int64(0)

In [ ]:
translation[
    'product_category_name'
].duplicated().sum()

np.int64(0)

clean text:


In [ ]:
translation['product_category_name'] = (
    translation['product_category_name']
    .str.strip()
    .str.lower()
)

translation['product_category_name_english'] = (
    translation['product_category_name_english']
    .str.strip()
    .str.title()
)

In [ ]:
# STEP 14 — MERGE CATEGORY TRANSLATION

products = products.merge(
    translation[
        ['product_category_name',
         'product_category_name_english']
    ],
    on='product_category_name',
    how='left'
)

# Drop original Portuguese category
products.drop(
    columns=['product_category_name'],
    inplace=True
)

# Rename English category
products.rename(
    columns={
        'product_category_name_english': 'product_category'
    },
    inplace=True
)

In [ ]:
products.head()

,product_id,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category
0,1e9e8ef04dbcff4541ed26657ea517e5,40.0,287.0,1.0,225.0,16.0,10.0,14.0,Perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,Art
2,96bd76ec8810374ed1b65e291975717f,46.0,250.0,1.0,154.0,18.0,9.0,15.0,Sports_Leisure
3,cef67bcfe19066a932b7673e239eb23d,27.0,261.0,1.0,371.0,26.0,4.0,26.0,Baby
4,9dc1a7de274444849c219cff195d0b71,37.0,402.0,4.0,625.0,20.0,17.0,13.0,Housewares


## STEP 15 — FINAL DATA QUALITY CHECK

In [ ]:
final_tables = {
    'customers': customers,
    'geolocation': geolocation,
    'items': items,
    'payments': payments,
    'reviews': reviews,
    'orders': orders,
    'products': products,
    'sellers': sellers,
    'translation': translation
}

for name, df in final_tables.items():

    print(f"\n========== {name.upper()} ==========")

    print("Rows:", len(df))
    print("Columns:", len(df.columns))
    print("Duplicate rows:", df.duplicated().sum())
    print("Total NULLs:", df.isnull().sum().sum())


========== CUSTOMERS ==========
Rows: 99441
Columns: 5
Duplicate rows: 0
Total NULLs: 0

========== GEOLOCATION ==========
Rows: 1000163
Columns: 5
Duplicate rows: 261831
Total NULLs: 0

========== ITEMS ==========
Rows: 112650
Columns: 7
Duplicate rows: 0
Total NULLs: 0

========== PAYMENTS ==========
Rows: 103886
Columns: 5
Duplicate rows: 0
Total NULLs: 0

========== REVIEWS ==========
Rows: 99224
Columns: 7
Duplicate rows: 0
Total NULLs: 0

========== ORDERS ==========
Rows: 96478
Columns: 16
Duplicate rows: 0
Total NULLs: 8

========== PRODUCTS ==========
Rows: 32951
Columns: 9
Duplicate rows: 0
Total NULLs: 2461

========== SELLERS ==========
Rows: 3095
Columns: 4
Duplicate rows: 0
Total NULLs: 0

========== TRANSLATION ==========
Rows: 71
Columns: 2
Duplicate rows: 0
Total NULLs: 0


## STEP 16 — SAVE CLEAN TABLES

In [ ]:
output_folder = 'clean_data'

os.makedirs(output_folder, exist_ok=True)

customers.to_csv(
    f'{output_folder}/customers.csv',
    index=False
)

geolocation.to_csv(
    f'{output_folder}/geolocation.csv',
    index=False
)

items.to_csv(
    f'{output_folder}/order_items.csv',
    index=False
)

payments.to_csv(
    f'{output_folder}/order_payments.csv',
    index=False
)

reviews.to_csv(
    f'{output_folder}/order_reviews.csv',
    index=False
)

orders.to_csv(
    f'{output_folder}/orders.csv',
    index=False
)

products.to_csv(
    f'{output_folder}/products.csv',
    index=False
)

sellers.to_csv(
    f'{output_folder}/sellers.csv',
    index=False
)

translation.to_csv(
    f'{output_folder}/category_translation.csv',
    index=False
)

print("\n✅ All 9 cleaned tables saved successfully.")


✅ All 9 cleaned tables saved successfully.


In [ ]:
translation.head()

,product_category_name,product_category_name_english
0,beleza_saude,Health_Beauty
1,informatica_acessorios,Computers_Accessories
2,automotivo,Auto
3,cama_mesa_banho,Bed_Bath_Table
4,moveis_decoracao,Furniture_Decor


In [ ]:
import pandas as pd

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:

    orders[col] = pd.to_datetime(
        orders[col],
        errors="coerce"
    )

In [ ]:
for col in date_columns:

    orders[col] = orders[col].dt.strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    orders[col] = orders[col].fillna("")

In [ ]:
for col in date_columns:

    print(
        col,
        "missing:",
        orders[col].isna().sum(),
        "zero dates:",
        orders[col].astype(str)
        .str.contains("0000-00-00", na=False)
        .sum()
    )

order_purchase_timestamp missing: 0 zero dates: 0
order_approved_at missing: 0 zero dates: 0
order_delivered_carrier_date missing: 0 zero dates: 0
order_delivered_customer_date missing: 0 zero dates: 0
order_estimated_delivery_date missing: 0 zero dates: 0


In [ ]:
orders.to_csv(
    r"D:\Financial-Dashboard-project\clean-data\clean_orders.csv",
    index=False
)